# HAM10000 class-conditional DDPM

Trains the generator and publishes synthetic df images for C4. The notebook covers setup, a small CPU pipeline check, training, and sample validation/publication.

Training uses all seven classes from the fixed train split. Generation selects df through class conditioning. Project and storage paths are configured in cell 1.3.


# Phase 1 · Setup

## 1.1 GPU check

Requires a GPU runtime.


In [ ]:
!nvidia-smi

## 1.2 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1.3 Project and data paths

`PROJECT_DIR` identifies the project directory on Drive.


In [ ]:
import os
from pathlib import Path

# Project directory on the mounted Drive.
PROJECT_DIR = '/content/drive/MyDrive/ddpm-derm-augmentation'
DATA_DIR    = PROJECT_DIR + '/data'
OUTPUTS_DIR = PROJECT_DIR + '/outputs'
LOCAL_CKPT_DIR = '/content/ddpm_ckpt'
SNAPSHOT_DIR = OUTPUTS_DIR + '/ddpm/checkpoints'

os.environ['DDPM_DERM_DATA_DIR'] = DATA_DIR
os.environ['DDPM_DERM_OUTPUTS_DIR'] = OUTPUTS_DIR
os.environ['DDPM_DERM_LOCAL_CKPT_DIR'] = LOCAL_CKPT_DIR

assert Path(PROJECT_DIR, 'src', 'ddpm_derm', 'config.py').is_file(), \
    f'PROJECT_DIR wrong: no src/ddpm_derm/config.py under {PROJECT_DIR}'
assert Path(DATA_DIR, 'manifests', 'class_to_idx.json').is_file(), \
    f'DATA_DIR wrong: no manifests/class_to_idx.json under {DATA_DIR}'
assert Path(SNAPSHOT_DIR).is_dir(), f'SNAPSHOT_DIR missing: {SNAPSHOT_DIR}'
print('paths OK')
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR    =', DATA_DIR)
print('OUTPUTS_DIR =', OUTPUTS_DIR)
print('LOCAL_CKPT_DIR =', LOCAL_CKPT_DIR)
print('SNAPSHOT_DIR =', SNAPSHOT_DIR)

## 1.4 Copy data to local disk (required for training)

Reading 10k small images directly from a shared Drive slows every epoch, so
copy data to Colab's local disk once per new runtime; the copy may print
nothing while it runs. Checkpoints remain on Drive.
Validation uses the fixed train/val/test manifests, so an extra unreferenced
JPG does not affect the experiment.

In [ ]:
import subprocess
import pandas as pd

DATA_DIR = '/content/data'

def _manifest_status():
    try:
        rows = pd.concat([
            pd.read_csv(Path(DATA_DIR, 'manifests', f'{split}.csv'))
            for split in ('train', 'val', 'test')
        ], ignore_index=True)
    except FileNotFoundError:
        return None, None
    missing = [p for p in rows['image_path'] if not Path(DATA_DIR, p).is_file()]
    return rows, missing

_rows, _missing = _manifest_status()
_ready = (_rows is not None and len(_rows) == 10015
          and _rows['image_path'].nunique() == 10015 and not _missing)
if _ready:
    print('local data already complete; skipping Drive copy')
else:
    print('copying data from Drive to /content/data (may be silent for a while) ...')
    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    subprocess.run(['cp', '-a', PROJECT_DIR + '/data/.', DATA_DIR + '/'], check=True)
    _rows, _missing = _manifest_status()

os.environ['DDPM_DERM_DATA_DIR'] = DATA_DIR
assert _rows is not None, 'local manifests are still missing after copy'
assert len(_rows) == 10015, f'unexpected manifest rows: {len(_rows)}'
assert _rows['image_path'].nunique() == 10015, 'manifest paths are not unique'
assert not _missing, f'missing manifest images: {len(_missing)}'
print('using local data:', DATA_DIR)
print('manifest rows: 10015; missing: 0')

## 1.5 Install deps

Colab already has torch/torchvision/pandas/pillow. Stage 2 adds **diffusers**
(the class-conditional `UNet2DModel`).

In [ ]:
!pip install -q diffusers

# Phase 2: Smoke test

Checks data loading, a small UNet training step, and DDIM sampling on CPU.


## 2.1 DDPM smoke test

In [ ]:
!cd "{PROJECT_DIR}" && python scripts/smoke_ddpm.py

# Phase 3 · Real run

## 3.1 Train the DDPM (target: epoch 100 for the formal C4 dataset)

`--resume` is the default: a missing checkpoint fails instead of restarting.
Remove it only for an intentional fresh run. The mutable `run_seed0_last.pt`
stays on local `/content` storage. Every 10 epochs and at the target epoch,
training writes a new immutable Drive snapshot, `run_seed0_epochNNNN.pt`,
without overwriting existing Drive files. A new runtime resumes from the
latest snapshot.

Checkpoints store raw and EMA weights (`--ema-decay 0.999`); previews use EMA.
The formal C4 dataset uses the epoch-100 snapshot, hence `--epochs 100`.


In [ ]:
# Safe default: resume only. Remove --resume only for an intentional fresh run.
!cd "{PROJECT_DIR}/src" && python -m ddpm_derm.train_ddpm --epochs 100 --img-size 64 --batch-size 64 --ema-decay 0.999 --preview-every 5 --resume --output-dir "{LOCAL_CKPT_DIR}" --snapshot-dir "{SNAPSHOT_DIR}" --snapshot-every 10

## 3.2 Formal epoch-100 sampling → validate → publish (manual cells)

The formal C4 dataset is **500 df images from the epoch-100 EMA snapshot**.
Sampling and publication are separate from setup and reject existing outputs.
The epoch-60 set directly under `outputs/synthetic_df/` is retained:

1. **Restore + stage** — copy the immutable Drive snapshot
   `run_seed0_epoch0100.pt` to local disk, then sample **500** df images with
   **50 DDIM steps, seed 0** onto fast local staging
   `/content/synthetic_df_epoch0100_seed0`. Aborts if the checkpoint is not
   epoch 100 with EMA weights, or if the staging dir is non-empty.
2. **Publish** — validate staging (row count, labels, portable relative paths,
   file presence, metadata), copy the **whole folder** to the new versioned
   Drive dir `outputs/synthetic_df/epoch0100_seed0`, re-validate on Drive, and
   only then write `_READY.json`. The C4 classifier run refuses to start
   without that marker. Images are never written to Drive one by one.
3. **Inspect** — show the final manifest count, missing-file count, metadata
   and `_READY` status.

If something fails partway: staging can be cleared with the commented
`rm -rf` line (local scratch only); a partial Drive folder has **no**
`_READY.json` — delete it manually in the Drive UI before retrying. Nothing
is deleted automatically.

In [ ]:
# [manual] 3.2a Restore the epoch-100 snapshot to local disk, then sample 500 df to staging.
import shutil
from pathlib import Path

EPOCH      = 100
SNAP       = Path(SNAPSHOT_DIR) / f'run_seed0_epoch{EPOCH:04d}.pt'
LOCAL_CKPT = Path(LOCAL_CKPT_DIR) / SNAP.name
STAGING    = '/content/synthetic_df_epoch0100_seed0'

assert SNAP.is_file(), (
    f'epoch-{EPOCH} snapshot not on Drive yet: {SNAP}\n'
    f'train to epoch {EPOCH} first (3.1); snapshots are written every 10 epochs')
Path(LOCAL_CKPT_DIR).mkdir(parents=True, exist_ok=True)
if not (LOCAL_CKPT.is_file() and LOCAL_CKPT.stat().st_size == SNAP.stat().st_size):
    print(f'copying snapshot to local disk: {SNAP} -> {LOCAL_CKPT}')
    shutil.copy2(SNAP, LOCAL_CKPT)   # copy only; the Drive snapshot stays immutable
print('local checkpoint ready:', LOCAL_CKPT)

# !rm -rf "{STAGING}"   # uncomment ONLY to clear a failed partial staging run

# Require epoch 100, EMA weights, and an empty output directory.
!cd "{PROJECT_DIR}/src" && python -m ddpm_derm.sample_ddpm --ckpt "{LOCAL_CKPT}" --require-epoch 100 --require-ema --n 500 --num-steps 50 --eta 0.0 --seed 0 --out-dir "{STAGING}"

In [ ]:
# [manual] 3.2b Validate staging -> copy whole folder to a NEW versioned Drive
# dir -> re-validate on Drive -> only then write _READY.json. Refuses an
# existing destination; a failed attempt leaves no _READY marker.
FINAL = OUTPUTS_DIR + '/synthetic_df/epoch0100_seed0'
!cd "{PROJECT_DIR}/src" && python -m ddpm_derm.publish_synthetic --src "{STAGING}" --dest "{FINAL}" --expect-n 500 --expect-epoch 100 --expect-seed 0 --expect-steps 50

In [ ]:
# [manual] 3.2c Final state of the published set (torch-free; safe to re-run).
import json
import pandas as pd
from pathlib import Path

final = Path(OUTPUTS_DIR) / 'synthetic_df' / 'epoch0100_seed0'
manifest = final / 'synthetic_df.csv'
assert manifest.is_file(), f'not published yet: no {manifest}'
rows = pd.read_csv(manifest)
missing = [p for p in rows['image_path'] if not (final / p).is_file()]
print('final dir     :', final)
print('manifest rows :', len(rows))
print('missing files :', len(missing))
print('metadata      :', json.dumps(json.loads((final / 'metadata.json').read_text()), indent=2))
ready = final / '_READY.json'
if ready.is_file():
    print('_READY.json   : PRESENT ->', json.loads(ready.read_text())['published_utc'])
else:
    print('_READY.json   : MISSING -> the set is NOT complete; do not train C4 on it')

## 3.3 Show previews + NN montage

Torch-free display from the saved PNGs — safe to re-run after a disconnect once
the setup cells (1.2, 1.3) have run. NN montages are **versioned per sampling
run** (`outputs/synthetic_df/<version>/nn_check.png`) and shown with their
version name. The old fixed-name montage `outputs/figures/ddpm_nn_check_df.png`
is the **epoch-60 legacy artifact**; nothing writes to it anymore.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

SAMPLES = Path(OUTPUTS_DIR) / 'ddpm' / 'samples'
SYN     = Path(OUTPUTS_DIR) / 'synthetic_df'

previews = sorted(SAMPLES.glob('preview_*.png'))
if previews:
    print('latest df preview grid:', previews[-1].name)
    display(Image(str(previews[-1])))
else:
    print('no preview grids yet -- run 3.1 with --preview-every > 0')

montages = sorted(SYN.glob('*/nn_check.png'))
if montages:
    for m in montages:
        print(f'NN check [{m.parent.name}] (left = generated, right = nearest real train df):')
        display(Image(str(m)))
else:
    print('no versioned NN montages yet -- run 3.2 first')
    print('(the legacy epoch-60 montage lives at outputs/figures/ddpm_nn_check_df.png)')

---